# Lecture 5


# Lecture Notes: Image Classification with Convolutional Neural Networks (CNNs)

---

## 1. Motivation: Beyond Flat Vectors

### The Problem with Fully Connected (FC) Networks
In standard Multilayer Perceptrons (MLPs) / Fully Connected Layers:
* A 2D image (e.g., $32 \times 32 \times 3$) is flattened into a 1D vector ($3072 \times 1$).
* **This destroys the spatial structure of images.** Pixels next to each other lose their spatial relationship because every pixel is treated as an independent dimension in a giant dot product ($W x$).
* A standard FC layer learns **one global template per class or per hidden neuron**.

```
Flattening: [32 x 32 x 3]  --->  [3072 x 1]  (Spatial relationships lost)
```

---

### Traditional Features vs. End-to-End Deep Learning
Before deep CNNs, computer vision relied on hand-engineered feature representations:
1. **Color Histograms:** Count frequency of colors, ignoring spatial arrangement.
2. **Histogram of Oriented Gradients (HoG):** Count edge directions in local $8 \times 8$ pixel grids.
3. **Bag of Words:** Extract random patches, cluster them into a "codebook" of visual words, and construct a histogram of word counts.

```
[ Image ] ---> [ Hand-crafted Feature Extractor ] ---> [ Linear Classifier ] ---> Scores
```

**The ConvNet Philosophy:**
Instead of fixed, hand-crafted features, train the **feature extractor and classifier together end-to-end** directly from raw pixels.

```
[ Image ] ---> [ Learnable Conv Layers (Feature Extractor) + Classifier ] ---> Scores
```

---

## 2. A Brief History of CNNs

* **Hubel & Wiesel (1959-1968):** Discovered the visual hierarchy in cat visual cortices:
  * **Simple Cells:** Respond to light orientations and edges.
  * **Complex Cells:** Respond to light orientation plus movement.
  * **Hypercomplex Cells:** Respond to movement with an endpoint.
  * **Topographical Mapping:** Nearby cells in the visual cortex represent nearby regions in the visual field.
* **Fukushima (1980) - Neocognitron:** Introduced the "sandwich" architecture alternating Simple ($S$) and Complex ($C$) cell layers. $S$-cells had learnable parameters; $C$-cells performed pooling.
* **LeCun et al. (1998) - LeNet-5:** Applied gradient-based learning (backpropagation) to stacked convolutions and subsampling for digit recognition.
* **Krizhevsky et al. (2012) - AlexNet:** Sparked the modern deep CNN era. Trained a massive network on ImageNet using GPUs, ReLUs, and Dropout.

---

## 3. The Convolutional Layer (CONV)

### Key Intuition & Structure
Unlike FC layers, a **Convolution Layer preserves 3D spatial structure** ($Width \times Height \times Depth$).

* **Filters (Kernels):** Small spatially (e.g., $5 \times 5$ or $3 \times 3$), but **always extend the full depth** of the input volume.
* **Operation:** Slide (convolve) the filter spatially across the input volume, computing a 3D dot product (+ bias) at each location.

$$\text{Output Value} = w^T x + b$$

```
Input Volume:  [32 x 32 x 3]
Filter:        [ 5 x  5 x 3]  (Depth 3 matches input depth 3)
Output:        1 scalar per spatial position -> forms a 2D Activation Map
```

If we use **$K$ different filters**, we get **$K$ activation maps** stacked together to form an output volume of depth $K$.

---

### Spatial Dimension Formulas (The Cheat Sheet)

Given:
* Input size: $W_1 \times H_1 \times C_{in}$
* Filter spatial size: $F \times F$
* Stride: $S$ (step size when sliding)
* Zero Padding: $P$ (border pixels added around input)
* Number of filters: $K$ ($C_{out}$)

#### 1. Output Spatial Dimensions ($W_2 \times H_2 \times K$):
$$W_2 = \frac{W_1 - F + 2P}{S} + 1$$
$$H_2 = \frac{H_1 - F + 2P}{S} + 1$$

> **Note:** If $(W_1 - F + 2P)$ is not evenly divisible by $S$, the stride does not fit properly.

#### 2. Number of Parameters:
$$\text{Parameters per filter} = F \cdot F \cdot C_{in} + 1 \quad (+1 \text{ for bias})$$
$$\text{Total Parameters} = K \cdot (F^2 \cdot C_{in} + 1)$$

---

### Worked Example

* **Input:** $32 \times 32 \times 3$
* **Filters:** $10$ filters of size $5 \times 5$, Stride $S = 1$, Padding $P = 2$

$$\text{Output Width} = \frac{32 - 5 + 2(2)}{1} + 1 = 32$$
$$\text{Output Volume} = 32 \times 32 \times 10$$

$$\text{Params per filter} = (5 \times 5 \times 3) + 1 = 76$$
$$\text{Total Params} = 76 \times 10 = 760 \text{ parameters}$$

---

### Special Case: $1 \times 1$ Convolutions
A $1 \times 1$ filter operates over a single spatial position ($1 \times 1$) across all input channels.
* Works as a **cross-channel linear combination** (a mini FC layer per pixel).
* Used for **dimensionality reduction** (reducing spatial depth/channel size) without altering height or width.


In [ ]:
# Conceptual 1x1 Conv in PyTorch
import torch.nn as nn

# Input: [Batch, 64, 56, 56] -> Output: [Batch, 32, 56, 56]
conv1x1 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=1, stride=1, padding=0)